# H14 walkthrough: a regime overlay over a trend model and a range model

This notebook runs the whole H14 study step by step, using the same code as `scripts/research/h14_study.py`:

1. design: the parameters and the pieces;
2. data and its checks;
3. the regime features and the trend probability;
4. the trend model and the range model;
5. the regime state machine and the model weights;
6. the backtest of every baseline, with equity curves;
7. the alternatives: simple filters, feature ablation, confidence scaling;
8. statistical testing with the strategy card;
9. a sandbox for your own overlay hypotheses.

The pre-registered result is in `research/reports/H14.md`: **D, not supported**. The primary overlay's net Sharpe was
0.19, against 0.29 for the static 50/50 blend. Use the sandbox to form a *new* hypothesis.

**Before testing anything new for real:**
- write it down as a new pre-registration (`python scripts/research/new_hypothesis.py new H15 "..."`);
- lock it, then run it once.

Everything you try in this notebook is a look at the data, and the more you try, the higher the bar. `RECORD = False`
keeps notebook cards out of the trial ledger.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.research import regime_switch as rs, strategy_checks as stc
plt.rcParams["figure.figsize"] = (14, 4)
RECORD = False
%matplotlib inline   # after the imports: some repo modules switch matplotlib to the non-interactive Agg backend

## 1. Design: every parameter in one file

All numbers live in `research/h14_regime_switch.toml`, and nothing is a hidden constant. `config.with_changes(...)`
gives a modified copy, which is how sections 7 and 9 try other settings. The defaults are the owner's starting
hypotheses, not fitted values.

In [ ]:
config = rs.load_config()
for section in ("trend", "range", "regime", "execution"):
    settings = getattr(config, section)
    print(f"[{section}]", {name: getattr(settings, name) for name in settings.__slots__})

## 2. Data and its checks

Daily index OHLC from Yahoo stands in for the futures: the Nasdaq-100 for NQ, plus six more markets. A rolled future
earns the index return plus (dividend yield − local 3-month rate) ÷ 252, so there are no roll gaps. Close-only early
history is cut, and 2026 stays frozen.

In [ ]:
rates = rs.load_rates()
MARKETS = [config.data["primary"], *config.data["cross_markets"]]
raw = {m: rs.load_market(config, m) for m in MARKETS}
frames = {m: f[f.index >= rs.usable_from(f)] for m, f in raw.items()}
pd.DataFrame({m: {**rs.data_checks(raw[m]), "usable from": f"{rs.usable_from(raw[m]):%Y-%m-%d}"} for m in MARKETS}).T

In [ ]:
MARKET = "NDX"                       # try "SPX", "DAX", "NIKKEI", ...
frame = frames[MARKET]
returns = rs.market_returns(config, MARKET, frame, rates)
START, END, SPLIT = pd.Timestamp(config.data["evaluation_start"]), pd.Timestamp(config.data["evaluation_end"]), pd.Timestamp(config.data["validation_start"])
frame["close"].loc[START:END].plot(logy=True, title=f"{MARKET} close (log)");

## 3. Regime features, scores and the trend probability

Each feature is known at its bar's close. Each becomes a percentile scored against the 500 days before the calendar
year, frozen within the year (walk-forward). The trend probability is
`0.25 ER + 0.25 slope + 0.20 ADX + 0.15 VR + 0.15 breakout`.

In [ ]:
scores = rs.regime_scores(frame, config)
fig, axes = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
for ax, name in zip(axes, ["er", "slope", "adx", "vr", "breakout"]):
    scores[f"{name}_score"].loc[START:END].plot(ax=ax, lw=0.5, title=f"{name} score")
scores["trend_probability"].loc[START:END].plot(ax=axes[-1], lw=0.6, title="trend probability")
for level in (config.regime.trend_entry, config.regime.range_entry, config.regime.exit_threshold):
    axes[-1].axhline(level, ls="--", c="grey")
plt.tight_layout()

In [ ]:
# How alike are the features? (rank correlation; ER and slope are nearly the same thing)
scores[["er", "abs_slope_t", "adx", "vr", "breakout_beyond", "vol_ratio"]].loc[START:END].corr(method="spearman").round(2)

## 4. The two models

- **Trend:** 0.5 × tanh((20-day MA − 100-day MA) ÷ ATR ÷ 2) + 0.5 × a 60/20-day breakout state, in [−1, 1].
- **Range:** z = (close − 40-day EWMA) ÷ residual standard deviation. It enters at |z| ≥ 1.5 and exits when z crosses
  zero, at the boundary stops, after 20 days, or on a breakout. Version A is symmetric, version B caps shorts at 0.5.

Standalone, neither depends on the overlay.

In [ ]:
trend = rs.trend_model(frame, config)
inputs = rs.range_inputs(frame, config)
range_alone, range_trades = rs.range_model(frame, inputs, trend, scores, config, regime=None, version="A")
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
trend["trend_signal"].loc[START:END].plot(ax=axes[0], lw=0.5, title="trend signal")
range_alone["range_signal"].loc[START:END].plot(ax=axes[1], lw=0.5, title="range signal (standalone)")
plt.tight_layout()
range_trades["reason"].value_counts()

## 5. The regime machine and the weights

- **Entering a regime:** TREND at ≥ 0.65, RANGE at ≤ 0.35, each after 3 confirmations.
- **Leaving a regime:** after 2 closes beyond 0.50. A move between TREND and RANGE goes through TRANSITION.
- **Transition weights:**
  - neutral TRANSITION: 0.25 / 0.25;
  - from RANGE to TREND: 0.25/0.25, then 0/0.5, then 0/1;
  - from TREND to RANGE: 0.5/0, then 0.25/0.25, then 1/0.
- **Rapid breakout:** goes straight to TRANSITION.
- **Stress:** halves the signal.

`variant=` picks the overlay: `blend` (primary), `hysteresis`, `hard`, `static`, `probability`, `filter`,
`trend_only`, `range_only`.

In [ ]:
regime = rs.classify(scores, config, variant="blend")
w = regime.loc[START:END]
fig, ax = plt.subplots(figsize=(14, 5))
close = frame["close"].loc[START:END]
ax.plot(close, c="k", lw=0.7); ax.set_yscale("log")
for state, color in {"RANGE": "#4C9AFF", "TRANSITION": "#FFC400", "TREND": "#36B37E"}.items():
    ax.fill_between(close.index, close.min(), close.max(), where=(w["state"] == state).to_numpy(), color=color, alpha=0.25, step="post", label=state)
ax.legend(); ax.set_title("Price with the overlay's regimes")
print(w["state"].value_counts(normalize=True).round(3).to_dict(), "| regime changes a year:",
      round(w["state"].ne(w["state"].shift()).sum() / (len(w) / 252), 1))

In [ ]:
# How well does each standalone model do on the days the overlay calls each regime? (the overlay's core claim)
trend_run = rs.backtest("trend", trend["trend_signal"], returns, config, cost_bps=config.markets[MARKET]["cost_bps"])
range_run = rs.backtest("range", range_alone["range_signal"], returns, config, cost_bps=config.markets[MARKET]["cost_bps"])
sharpe = lambda s: s.mean() / s.std() * np.sqrt(252) if s.std() > 0 else np.nan
pd.DataFrame({name: {state: sharpe(run.net.loc[START:END][w["state"] == state]) for state in ("RANGE", "TRANSITION", "TREND")}
              for name, run in (("trend model", trend_run), ("range model", range_run))}).T.round(2)

## 6. Every baseline, backtested

`rs.run_market` runs every predefined strategy:
- positions are signal × 10% ÷ volatility, with a 2% band;
- a signal from the close of t trades at the close of t+1 and earns from t+2;
- costs are per side plus rolls.

It takes a few seconds.

In [ ]:
study = rs.run_market(config, MARKET, frame, rates, full=True)
CORE = ["1 trend alone", "2A range alone", "3A hard switch", "4A three-state, hysteresis", rs.PRIMARY, "6A static 50/50",
        "7A 200-day MA regime filter", "9A exposure-matched static", "8 buy and hold (1x)", "8v volatility-managed long"]
table = pd.DataFrame({n: {**rs.metrics(study.runs[n], START, END), "Sharpe exploratory": rs.metrics(study.runs[n], START, SPLIT - pd.Timedelta(days=1))["Sharpe"],
                          "Sharpe validation": rs.metrics(study.runs[n], SPLIT, END)["Sharpe"]} for n in CORE}).T
table[["annual return", "volatility", "Sharpe", "Sharpe exploratory", "Sharpe validation", "max drawdown", "turnover", "time invested", "long contribution", "short contribution"]].round(3)

In [ ]:
def equity_curves(runs, names, title="Equity curves (net)"):
    fig, (ax, dd) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    for n in names:
        wealth = (1 + runs[n].net.loc[START:END]).cumprod()
        ax.plot(wealth, label=n, lw=1.8 if n == rs.PRIMARY else 0.9)
        dd.plot(wealth / wealth.cummax() - 1, lw=0.8)
    ax.set_yscale("log"); ax.legend(fontsize=8); ax.set_title(title); dd.set_title("Drawdown")
    plt.tight_layout()

equity_curves(study.runs, [n for n in CORE if not n.startswith("8 buy")])

## 7. Against the alternatives

These are the comparisons a regime overlay has to win:
- simple filters (each a hard switch: trend model when the filter is on, range model otherwise);
- the overlay without each feature, and with each feature alone;
- confidence scaling (trend weight = trend probability, no states).

In [ ]:
alt = [rs.PRIMARY, "6A static 50/50", "P probability-weighted (confidence scaling)", "7A 200-day MA regime filter",
       *[n for n in study.runs if n.startswith(("F ", "X ", "S "))]]
pd.DataFrame({n: rs.metrics(study.runs[n], START, END) for n in alt}).T[["Sharpe", "annual return", "max drawdown", "turnover"]].sort_values("Sharpe", ascending=False).round(3)

In [ ]:
# Sensitivity: change one parameter at a time and compare with the static blend at the same setting
def primary_and_static(cfg):
    s = rs.run_market(cfg, MARKET, frame, rates, full=False)
    return rs.metrics(s.runs[rs.PRIMARY], START, END)["Sharpe"], rs.metrics(s.runs["6A static 50/50"], START, END)["Sharpe"]

trials = {"baseline": {}, "regime horizon 60": {"regime": {"horizon": 60}}, "TREND entry 0.60": {"regime": {"trend_entry": 0.60}},
          "confirmation 5": {"regime": {"confirm": 5}}, "range |z| 2.0": {"range": {"entry_z": 2.0}}}
pd.DataFrame({label: dict(zip(["primary Sharpe", "static Sharpe"], primary_and_static(config.with_changes(**changes) if changes else config)))
              for label, changes in trials.items()}).T.round(3)

## 8. Statistical testing: the strategy card

These checks decide whether a result is an edge or a fluke:
- the Newey-West t-stat and a block-bootstrap Sharpe interval;
- the deflated Sharpe, counting every variant tried;
- Romano-Wolf p-values across the variants;
- the timing placebo (positions shifted in time);
- a paired test against the baseline;
- a Bayesian shrunk Sharpe.

The pass marks are in `research/strategy_standard.toml`.

In [ ]:
net = lambda n: study.runs[n].net.loc[START:END]
variants = pd.DataFrame({n: net(n) for n in study.runs if not n.startswith(("8", "R "))})
card = stc.strategy_card(net(rs.PRIMARY), name=f"{rs.PRIMARY} on {MARKET}", baseline=net("6A static 50/50"), variants=variants, periods_per_year=252,
                         positions=study.runs[rs.PRIMARY].earning.loc[START:END].to_frame(MARKET), asset_returns=returns.loc[START:END].fillna(0).to_frame(MARKET),
                         ledger=RECORD)
print(card.verdict, "-", card.reason, "| against the static blend:", card.versus_baseline)
card.table

In [ ]:
card.plot();

## 9. Sandbox: your own overlay

Three ways in, from least to most work.

**a) Change parameters:** `config.with_changes(regime={...}, range={...}, trend={...})`.

**b) Your own trend probability:** any series in [0, 1] known at each close. It replaces the blended score, and the
same state machine, weights and range rules run on it.

**c) Your own regime rule:** a True/False series (True = TREND) used as a hard switch, or a pair of weight series
that you set yourself.

Every result goes through the same backtest and the same strategy card, against the static blend.

In [ ]:
COST = config.markets[MARKET]["cost_bps"]

def run_overlay(label, *, probability=None, trend_flag=None, weights=None, cfg=config, version="A"):
    """Backtest one overlay idea and return its run. Give exactly one of probability / trend_flag / weights
    (weights: a DataFrame with trend_weight and range_weight columns, values in [0, 1])."""
    s = rs.regime_scores(frame, cfg) if cfg is not config else scores.copy()
    t = rs.trend_model(frame, cfg) if cfg is not config else trend
    i = rs.range_inputs(frame, cfg) if cfg is not config else inputs
    if probability is not None:
        s["trend_probability"] = probability
        reg = rs.classify(s, cfg, variant="blend")
    elif trend_flag is not None:
        reg = rs.classify(s, cfg, variant="filter", filter_trend=trend_flag)
    elif weights is not None:
        reg = rs.classify(s, cfg, variant="static")
        reg["trend_weight"], reg["range_weight"] = weights["trend_weight"].reindex(reg.index).fillna(0), weights["range_weight"].reindex(reg.index).fillna(0)
    else:
        reg = rs.classify(s, cfg, variant="blend")
    overlay = reg if (probability is not None or trend_flag is not None or weights is None) else None
    ranged, _ = rs.range_model(frame, i, t, s, cfg, regime=overlay, version=version)
    target = rs.combine(t, ranged["range_signal"], reg, s, cfg)
    return rs.backtest(label, target["target_signal"], returns, cfg, cost_bps=COST)

def judge(run, baseline="6A static 50/50"):
    c = stc.strategy_card(run.net.loc[START:END], name=run.name, baseline=net(baseline), periods_per_year=252,
                          positions=run.earning.loc[START:END].to_frame(MARKET), asset_returns=returns.loc[START:END].fillna(0).to_frame(MARKET), ledger=RECORD)
    return pd.Series(c.summary()).to_frame(run.name).T.round(3)

In [ ]:
# Example b): one feature as the trend probability (slope alone did best in the ablation)
mine = run_overlay("slope-only probability", probability=scores["slope_score"])
# Example c): trend when the 20-day MA is above the 100-day MA (a simple filter)
flag = frame["close"].rolling(20).mean() > frame["close"].rolling(100).mean()
simple = run_overlay("20/100 MA switch", trend_flag=flag)
# Example: continuous weights of your own, e.g. the trend weight rising with the slope score and no range model at all
own = run_overlay("slope-weighted trend only", weights=pd.DataFrame({"trend_weight": scores["slope_score"].fillna(0), "range_weight": 0.0}))
pd.concat([judge(mine), judge(simple), judge(own)])

In [ ]:
equity_curves({**study.runs, mine.name: mine, simple.name: simple, own.name: own},
              [rs.PRIMARY, "6A static 50/50", "1 trend alone", mine.name, simple.name, own.name], "Your overlays against the baselines")

In [ ]:
# Does the idea travel? The same overlay on every market (unchanged rules), against the static blend there
def across_markets(make_flag):
    rows = {}
    for m in MARKETS:
        f, r = frames[m], rs.market_returns(config, m, frames[m], rates)
        s, t, i = rs.regime_scores(f, config), rs.trend_model(f, config), rs.range_inputs(f, config)
        reg = rs.classify(s, config, variant="filter", filter_trend=make_flag(f))
        ranged, _ = rs.range_model(f, i, t, s, config, regime=reg)
        run = rs.backtest(m, rs.combine(t, ranged["range_signal"], reg, s, config)["target_signal"], r, config, cost_bps=config.markets[m]["cost_bps"])
        static = rs.run_market(config, m, f, rates, full=False).runs["6A static 50/50"]
        first = max(START, s["trend_probability"].first_valid_index())
        rows[m] = {"overlay Sharpe": rs.metrics(run, first, END)["Sharpe"], "static Sharpe": rs.metrics(static, first, END)["Sharpe"]}
    return pd.DataFrame(rows).T.round(3)

across_markets(lambda f: f["close"].rolling(20).mean() > f["close"].rolling(100).mean())

**Reading the sandbox fairly.** An idea that looks good here has been chosen after looking. Before believing it:
- check it survives in the validation period on its own (2012–2025), and on other markets with unchanged rules;
- check it beats the static blend on the paired test, not just on point Sharpe;
- check the timing placebo passes, so it isn't just being long a rising index;
- then pre-register it as a new hypothesis and run it once.